# Exploratory Data Analysis (EDA) — Refined

## Project Overview
This is a **trimmed-down version** of the original EDA notebook, refined specifically for portfolio use.

The full exploratory work (11 charts + extra tables) has been narrowed to the **7 charts that carry the most signal** for a public-facing Streamlit dashboard, plus a **Key Metrics** summary that maps directly to dashboard metric cards.

### What changed from the original notebook
**Kept (dashboard-ready):**
- Key Metrics summary (new)
- Salary Distribution
- Salary by Job Role
- Job Role Distribution
- Top Skills Required
- Top Hiring States
- Experience Required
- Remote vs In-Person

**Removed from the dashboard set (kept as supplementary analysis only, at the bottom):**
- Correlation Heatmap — too technical for a general dashboard audience
- Highest Paying Roles / States tables — not visual, redundant with the boxplot and states chart

**Dropped entirely:**
- Company Age Distribution — weak signal, low interpretive value
- Seniority Level Distribution — mostly "Not Specified", overlaps with Job Role chart
- Duplicate skills table — repeated the skills bar chart

Each chart below is tagged so you know exactly what to port into `app.py` for the Streamlit dashboard.


## Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")
pd.set_option("display.max_columns", None)


## Load Dataset

Update the path below to point to your `featured_jobs.csv` output from the feature engineering notebook.

In [ ]:
df = pd.read_csv("featured_jobs1.csv")
df.head()


## Color Palette

Streamlined to the shades actually used below.

In [ ]:
purple_colors = {
    "dark_purple": "#4B0082",
    "mauve": "#9370DB",
    "light_violet": "#C8A2C8",
}


## 📊 Key Metrics (Dashboard Header Cards)

These four numbers are what viewers see first — perfect for `st.metric()` cards at the top of the Streamlit app.

In [ ]:
total_jobs = len(df)
avg_salary = round(df["Avg_Salary"].mean(), 1)
avg_rating = round(df["Rating"].mean(), 2)
pct_remote = round(df["Remote_Job"].mean() * 100, 1)
top_skill = (
    df.filter(regex="_yn$")
    .sum()
    .sort_values(ascending=False)
    .index[0]
    .replace("_yn", "")
)

print(f"Total Job Listings : {total_jobs}")
print(f"Average Salary (K USD) : {avg_salary}")
print(f"Average Company Rating : {avg_rating}")
print(f"Percent Remote : {pct_remote}%")
print(f"Top Requested Skill : {top_skill}")


## 📊 Chart 1 — Salary Distribution
*Dashboard-ready*

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    df["Avg_Salary"],
    bins=30,
    color=purple_colors["light_violet"],
    kde=True
)

plt.title("Distribution of Average Salary", color=purple_colors["dark_purple"])
plt.xlabel("Average Salary (K USD)")
plt.ylabel("Count")
plt.show()


## 📊 Chart 2 — Salary by Job Role
*Dashboard-ready — strongest insight chart*

In [ ]:
plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="Job_Simplified",
    y="Avg_Salary",
    color=purple_colors["light_violet"]
)

plt.xticks(rotation=45)
plt.title("Average Salary by Job Role", color=purple_colors["dark_purple"])
plt.show()


## 📊 Chart 3 — Job Role Distribution
*Dashboard-ready*

In [ ]:
plt.figure(figsize=(12, 6))

sns.countplot(
    data=df,
    y="Job_Simplified",
    order=df["Job_Simplified"].value_counts().index,
    color=purple_colors["light_violet"]
)

plt.title("Job Role Distribution", color=purple_colors["dark_purple"])
plt.show()


## 📊 Chart 4 — Top Skills Required
*Dashboard-ready — classic, high-impact chart for this project type*

In [ ]:
skill_columns = [col for col in df.columns if col.endswith("_yn")]

skill_counts = (
    df[skill_columns]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
skill_counts.columns = ["Skill", "Count"]
skill_counts["Skill"] = skill_counts["Skill"].str.replace("_yn", "")

plt.figure(figsize=(12, 7))

bar = sns.barplot(
    data=skill_counts,
    x="Count",
    y="Skill",
    color=purple_colors["light_violet"]
)

for i, value in enumerate(skill_counts["Count"]):
    bar.text(value + 2, i, str(value), va="center")

plt.title("Most Requested Skills", color=purple_colors["dark_purple"])
plt.show()


## 📊 Chart 5 — Top Hiring States
*Dashboard-ready*

In [ ]:
plt.figure(figsize=(10, 8))

sns.countplot(
    data=df,
    y="Job_State",
    order=df["Job_State"].value_counts().head(15).index,
    color=purple_colors["mauve"]
)

plt.title("Top Hiring States", color=purple_colors["dark_purple"])
plt.show()


## 📊 Chart 6 — Experience Required
*Dashboard-ready*

In [ ]:
experience_counts = (
    df["Experience_Group"]
    .value_counts()
    .reindex(["0-2 Years", "3-5 Years", "6-10 Years", "10+ Years", "Unknown"])
)

plt.figure(figsize=(9, 5))

bar = sns.barplot(
    x=experience_counts.index,
    y=experience_counts.values,
    color=purple_colors["light_violet"]
)

for i, value in enumerate(experience_counts.values):
    bar.text(i, value, str(value), ha="center", va="bottom")

plt.title("Experience Required", color=purple_colors["dark_purple"])
plt.show()


## 📊 Chart 7 — Remote vs In-Person
*Dashboard-ready*

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="Remote_Job",
    color=purple_colors["light_violet"]
)

plt.xticks([0, 1], ["In-Person", "Remote"])
plt.title("Remote vs In-Person Jobs", color=purple_colors["dark_purple"])
plt.show()


---
## 🔍 Supplementary Analysis (Notebook Only — Not for Dashboard)

The items below are useful for deeper analysis but are either too technical or too redundant to include as dashboard tiles.

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))

numeric = df.select_dtypes(include="number")

sns.heatmap(
    numeric.corr(),
    cmap="Purples",
    annot=True,
    fmt=".2f"
)

plt.title("Correlation Heatmap", color=purple_colors["dark_purple"])
plt.show()


### Highest Paying Job Roles

In [ ]:
(
    df.groupby("Job_Simplified")["Avg_Salary"]
    .mean()
    .sort_values(ascending=False)
)


### Highest Paying States

In [ ]:
(
    df.groupby("Job_State")["Avg_Salary"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)


## Summary

### Final Dashboard Chart Set (7 + Key Metrics)
1. Key Metrics — total listings, avg salary, avg rating, % remote, top skill
2. Salary Distribution
3. Salary by Job Role
4. Job Role Distribution
5. Top Skills Required
6. Top Hiring States
7. Experience Required
8. Remote vs In-Person

### Kept for analysis only (not in dashboard)
- Correlation Heatmap
- Highest Paying Roles / States tables

### Dropped
- Company Age Distribution
- Seniority Level Distribution
- Duplicate skills table

This refined set keeps the dashboard focused and lets each chart earn its place — ready to port into `app.py` when you build the Streamlit dashboard.
